In [13]:
import pandas as pd
import os

# === CONFIGURATION ===
INPUT_CSV = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Sorted_URL_List.csv"
OUTPUT_CSV = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos.csv"

# === LOAD CSV ===
df = pd.read_csv(INPUT_CSV)

# === EXTRACT COMPONENTS IN LOWERCASE ===
df['clone_url'] = df['clone_url'].astype(str)
df['username'] = df['clone_url'].apply(lambda x: x.split('/')[-2].lower())

# Remove `.git` or anything after the first dot from project name
df['project_name'] = df['clone_url'].apply(
    lambda x: x.split('/')[-1].lower().split('.')[0]
)

# Combine for full name
df['full_name'] = df['username'] + '.' + df['project_name']

# Reset index to make it a column
df = df.reset_index()

# === REORDER: index + clone_url first ===
df_clean = df[['index', 'clone_url', 'username', 'project_name', 'full_name']]
df_clean.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Cleaned repo list saved to: {OUTPUT_CSV}")


✅ Cleaned repo list saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos.csv


In [14]:
import pandas as pd
import os

# === CONFIGURATION ===
output_dir = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output"
output_csv = os.path.join(output_dir, "7.4-Total_Repos.csv")

# === INPUT FILES ===
df_total_repos = pd.read_csv(output_csv)
df_gradle = pd.read_csv(os.path.join(output_dir, "7.1-Gradle List_4th_attempt_ShallowC.csv"))
df_project_list = pd.read_csv(os.path.join(output_dir, "7.3-Project_List_4thAttempt_ShallowC.csv"))

# === MERGE BY 'full_name' ===

df_merged = pd.merge(df_total_repos, df_gradle, on='full_name', how='left')
df_merged = pd.merge(df_merged, df_project_list, on='full_name', how='left')


# === CLEAN-UP COLUMN NAMES ===
df_merged['username'] = df_merged['username_x'].combine_first(df_merged['username_y'])
df_merged['project_name'] = df_merged['project_name_x'].combine_first(df_merged['project_name_y'])

# Drop the redundant columns
df_merged.drop(columns=['username_x', 'username_y', 'project_name_x', 'project_name_y'], inplace=True)

# Reorder columns (optional)
ordered_cols = ['username', 'project_name', 'full_name', 'clone_url'] + \
               [col for col in df_merged.columns if col not in ['username', 'project_name', 'full_name', 'clone_url']]

df_merged = df_merged[ordered_cols]

# === EXPORT FINAL CSV ===
df_merged.to_csv(output_csv, index=False)
print(f"✅ Final merged file saved to: {output_csv}")


✅ Final merged file saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos.csv


In [ ]:
#append the metadata to the total repos file

import pandas as pd

# === File Paths ===
total_repos_path = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos.csv"
metadata_path = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\8.2-Project_Metadata.csv"
output_path = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos_WithMetrics.csv"

# === Load CSVs ===
df_repos = pd.read_csv(total_repos_path)
df_metadata = pd.read_csv(metadata_path)

# === Fix full_name in metadata by replacing '/' with '.' ===
if 'full_name' in df_metadata.columns:
    df_metadata['full_name'] = df_metadata['full_name'].str.replace('/', '.', regex=False)
else:
    raise KeyError("❌ Column 'full_name' not found in metadata file.")

# === Select only relevant metrics ===
metrics_cols = [
    'language', 'license', 'created_at', 'updated_at', 'last_commit_date',
    'stars', 'forks', 'watchers', 'open_issues', 'contributors',
    'pull_requests', 'commits_GitAPI',  'size'
]
df_metadata_subset = df_metadata[['full_name'] + metrics_cols]

# === Merge on full_name using left join ===
df_enriched = pd.merge(df_repos, df_metadata_subset, how='left', on='full_name')

# === Save output ===
df_enriched.to_csv(output_path, index=False)

print(f"✅ Enriched repo data saved to: {output_path}")


✅ Enriched repo data saved to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos_WithMetrics.csv


In [16]:
import pandas as pd
import os

# === Paths ===
input_path = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos_WithMetrics.csv"
output_path = r"D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos_WithMetrics.csv"

# === Load CSV ===
df = pd.read_csv(input_path)

# === Rename columns
df.rename(columns={
    "Unit Test": "Unit_Test_CI",
    "Instrumentation Testing": "Instrumentation_Test_CI"
}, inplace=True)

# === Add new columns
df["Unit_Test(Local or CI)"] = df[["Unit_Test_CI", "has_local_unit_test"]].any(axis=1)
df["Instrumentation_Test(Local or CI)"] = df[["Instrumentation_Test_CI", "has_local_instrumentation_test"]].any(axis=1)

# === Clean Test_Type_Grouped
df["Test_Type_Group"] = df["Test_Type_Group"].str.replace(", none", "", regex=False)
df["Test_Type_Group"] = df["Test_Type_Group"].str.replace("none ,", "", regex=False)

# === Reorder to insert new columns before "GitHub Action"
insert_at = df.columns.get_loc("GitHub Action")
cols = list(df.columns)
# Remove if already at the end
cols.remove("Unit_Test(Local or CI)")
cols.remove("Instrumentation_Test(Local or CI)")
# Insert in order
cols.insert(insert_at, "Instrumentation_Test(Local or CI)")
cols.insert(insert_at, "Unit_Test(Local or CI)")
df = df[cols]

# === Save result
df.to_csv(output_path, index=False)
print(f"✅ Saved updated file to: {output_path}")


✅ Saved updated file to: D:\Android_Mobile_App\AndroidProject_4th\6.2-Shallow_Clone_Complete\Analysis Output\7.4-Total_Repos_WithMetrics.csv
